# Tahim Bhuiya

# Logistic Regression & Regularization

In [1]:
import pandas as pd
df = pd.read_csv('Airline_Delay_Cause.csv')
df

,year,month,carrier,carrier_name,airport,airport_name,arr_flights,arr_del15,carrier_ct,weather_ct,...,security_ct,late_aircraft_ct,arr_cancelled,arr_diverted,arr_delay,carrier_delay,weather_delay,nas_delay,security_delay,late_aircraft_delay
0,2022,5,9E,Endeavor Air Inc.,ABE,"Allentown/Bethlehem/Easton, PA: Lehigh Valley ...",136.0,7.0,5.95,0.00,...,0.0,1.00,0.0,0.0,255.0,222.0,0.0,4.0,0.0,29.0
1,2022,5,9E,Endeavor Air Inc.,ABY,"Albany, GA: Southwest Georgia Regional",91.0,16.0,7.38,0.00,...,0.0,6.09,0.0,0.0,884.0,351.0,0.0,81.0,0.0,452.0
2,2022,5,9E,Endeavor Air Inc.,ACK,"Nantucket, MA: Nantucket Memorial",19.0,2.0,0.13,0.00,...,0.0,0.88,1.0,0.0,138.0,4.0,0.0,106.0,0.0,28.0
3,2022,5,9E,Endeavor Air Inc.,AEX,"Alexandria, LA: Alexandria International",88.0,14.0,7.26,0.76,...,0.0,1.64,0.0,0.0,947.0,585.0,35.0,125.0,0.0,202.0
4,2022,5,9E,Endeavor Air Inc.,AGS,"Augusta, GA: Augusta Regional at Bush Field",181.0,19.0,13.84,0.00,...,0.0,2.09,0.0,0.0,808.0,662.0,0.0,87.0,0.0,59.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
318012,2003,6,DL,Delta Air Lines Inc.,SEA,"Seattle, WA: Seattle/Tacoma International",480.0,84.0,25.69,3.09,...,0.0,21.26,0.0,0.0,3744.0,978.0,164.0,1023.0,0.0,1579.0
318013,2003,6,DL,Delta Air Lines Inc.,SFO,"San Francisco, CA: San Francisco International",505.0,111.0,21.78,2.24,...,0.0,13.17,3.0,0.0,4284.0,1376.0,138.0,2132.0,0.0,638.0
318014,2003,6,DL,Delta Air Lines Inc.,SJC,"San Jose, CA: Norman Y. Mineta San Jose Intern...",146.0,36.0,6.99,0.00,...,0.0,2.15,0.0,0.0,896.0,205.0,0.0,607.0,0.0,84.0
318015,2003,6,DL,Delta Air Lines Inc.,SJU,"San Juan, PR: Luis Munoz Marin International",95.0,13.0,3.66,0.00,...,0.0,1.42,0.0,0.0,367.0,120.0,0.0,210.0,0.0,37.0


In [2]:
df.columns # print the columns

Index(['year', 'month', 'carrier', 'carrier_name', 'airport', 'airport_name',
       'arr_flights', 'arr_del15', 'carrier_ct', 'weather_ct', 'nas_ct',
       'security_ct', 'late_aircraft_ct', 'arr_cancelled', 'arr_diverted',
       'arr_delay', 'carrier_delay', 'weather_delay', 'nas_delay',
       'security_delay', 'late_aircraft_delay'],
      dtype='object')

In [3]:
df.shape # print the number of rows and columns

(318017, 21)

In [4]:
df = df.dropna() # remove rows containing nan values

In [5]:
df = df.drop_duplicates(keep='first') # drop duplicated rows, keep the first occurrence

In [6]:
df.shape

(317261, 21)

# Defining the Prediction Task

## A. Categorical outcome variable 

Choosing a categorical outcome variable and in a markdown cell state:

• What the variable is

• What “1” means

• What “0” means





**Outcome:** delay_category – whether a flight has a high arrival delay  
- **1 = High delay** (flight’s arrival delay is above the median, arr_del15 > median)  
- **0 = Low delay** (flight’s arrival delay is at or below the median, arr_del15 <= median)  

This defines a binary classification problem: predicting whether a flight is likely to experience a high delay.

## B. Predictor variables 

List at least predictor variables and explaining why each one
may help predict the outcome.

1. **month** – Problems such as seasonal conditions can increase the likelihood of delays.  
2. **arr_flights** – Total flights at the airport may indicate congestion which can lead to delays.  
3. **carrier_ct** – Number of delays caused by the carrier. Higher counts can indicate issues.  
4. **weather_category** – Newly created high/low weather-delay category. This indicates high weather delays which can help in predicting. 
5. **nas_delay** – Delays due to the National Airspace System. This can affect arrival times.  
6. **late_aircraft_ct** – Delays caused by the aircraft arriving late from previous flights. This may be indicative of more delays.
7. **arr_delay** – This is the total delay time and can serve as strong indicator of delay severity.

In [7]:
import pandas as pd

# Create high/low delay category based on median of arr_del15
delay_threshold = df['arr_del15'].median()
df['delay_category'] = df['arr_del15'] > delay_threshold  # True = High delay, False = Low delay

# Crosstab of carrier_name vs delay_category
observations = pd.crosstab(index=df["carrier_name"], columns=df["delay_category"], margins=True)
observations

delay_category,False,True,All
carrier_name,,,
ATA Airlines d/b/a ATA,478,440,918
AirTran Airways Corporation,2630,4325,6955
Alaska Airlines Inc.,8614,4319,12933
Allegiant Air,5484,885,6369
Aloha Airlines Inc.,159,94,253
America West Airlines Inc.,668,911,1579
American Airlines Inc.,6357,13413,19770
American Eagle Airlines Inc.,5644,9755,15399
Atlantic Coast Airlines,512,741,1253


In [8]:
# Create high/low weather-delay category based on median of weather_ct that will be used as a feature
weather_threshold = df['weather_ct'].median()
df['weather_category'] = df['weather_ct'] > weather_threshold  # True = High weather delay, False = Low

# Data Preparation

Preparing the dataset for modeling:

• Converting categorical variables into numeric form 

• Handling missing values 

• Separating the data into X (features) and y (outcome) 

• Splitting the data into training and test sets 

• Balancing the training data 

In [9]:
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mean_squared_error
from sklearn import preprocessing

## Converting categorical variables into numeric form

In [10]:
# Convert weather_category (True/False) → (1/0)
df['weather_category'] = df['weather_category'].astype(int)

## Handle missing values

Missing values (NaNs) have already been removed at beginning of notebook

Was done with:

df = df.dropna() # remove rows containing nan values

df = df.drop_duplicates(keep='first') # drop duplicated rows, keep the first occurrence

## Separate the data into X (features) and y (outcome)

In [11]:
# Select X and y
X = df[['month', 'arr_flights', 'carrier_ct', 'nas_delay', 
            'late_aircraft_ct', 'security_delay', 'weather_category']]
y = df['delay_category'].astype(int)

## Split the data into training and test sets

In [12]:
# Split data into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

## Balance the training data

In [13]:
train_df = pd.concat([X_train.reset_index(drop=True), y_train.reset_index(drop=True)], axis=1) # Combining to balance together

In [14]:
train_df['delay_category'].value_counts()

delay_category
0    130105
1    123703
Name: count, dtype: int64

In [15]:
# 0 is majority, 1 is minority
majority_class = train_df[train_df['delay_category'] == 0]
minority_class = train_df[train_df['delay_category'] == 1]

# Undersample majority to match minority size
majority_class_undersampled = majority_class.sample(n=len(minority_class), random_state=42)

# Combine undersampled majority with full minority
train_balanced = pd.concat([majority_class_undersampled, minority_class])
train_balanced = train_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

X_train_balanced = train_balanced.drop(columns=['delay_category']) # Separating after balancing
y_train_balanced = train_balanced['delay_category']

In [16]:
train_balanced['delay_category'].value_counts()

delay_category
1    123703
0    123703
Name: count, dtype: int64

In [17]:
from sklearn.preprocessing import StandardScaler

# Fit on training data only
scaler = StandardScaler().fit(X_train_balanced)

# Transform both datasets using the same scaler
X_train_scaled = scaler.transform(X_train_balanced)
X_test_scaled = scaler.transform(X_test)

# Logistic Regression with 5-Fold Cross-Validation

## A. Cross-validation - using only the training data

Training a logistic regression model using 5-fold cross-validation and reporting the average
performance across folds using the following metrics: accuracy, precision, recall, F1

In [18]:
# Initialize Logistic Regression
logreg = LogisticRegression()

# Accuracy
accuracy_scores = cross_val_score(logreg, X_train_scaled, y_train_balanced, cv=5, scoring='accuracy')
print("Logistic Regression - Accuracy Scores:", accuracy_scores)
print("Mean Accuracy:", accuracy_scores.mean())

# Precision
precision_scores = cross_val_score(logreg, X_train_scaled, y_train_balanced, cv=5, scoring='precision')
print("\nLogistic Regression - Precision Scores:", precision_scores)
print("Mean Precision:", precision_scores.mean())

# Recall
recall_scores = cross_val_score(logreg, X_train_scaled, y_train_balanced, cv=5, scoring='recall')
print("\nLogistic Regression - Recall Scores:", recall_scores)
print("Mean Recall:", recall_scores.mean())

# F1 Score
f1_scores = cross_val_score(logreg, X_train_scaled, y_train_balanced, cv=5, scoring='f1')
print("\nLogistic Regression - F1 Scores:", f1_scores)
print("Mean F1 Score:", f1_scores.mean())

Logistic Regression - Accuracy Scores: [0.97421285 0.97378792 0.97550575 0.97403044 0.97297953]
Mean Accuracy: 0.9741032953543984

Logistic Regression - Precision Scores: [0.97757154 0.9767754  0.97836783 0.9772903  0.97864769]
Mean Precision: 0.9777305512685084

Logistic Regression - Recall Scores: [0.97069641 0.970656   0.97251526 0.97061439 0.9670574 ]
Mean Recall: 0.9703078911263617

Logistic Regression - F1 Scores: [0.97412185 0.97370608 0.97543276 0.97394091 0.97281802]
Mean F1 Score: 0.9740039242733743


## B. Final model 

After cross-validation, retrain the logistic regression model on the entire training set and
evaluate it on the test set. Report the same performance metrics and print out model
coefficients.

In [19]:
from sklearn.metrics import classification_report

# Train final logistic regression model
final_model = LogisticRegression()
final_model.fit(X_train_scaled, y_train_balanced)

# Print coefficients with feature names
print("Model Coefficients:")
for feature, coef in zip(X.columns, final_model.coef_[0]): # Using to make coefficients show up neatly
    print(f"{feature}: {coef:.4f}")

# Predict on the test set
y_pred = final_model.predict(X_test_scaled)

# Classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Model Coefficients:
month: -0.0195
arr_flights: 4.5190
carrier_ct: 26.8174
nas_delay: 47.1367
late_aircraft_ct: 37.0990
security_delay: 0.3130
weather_category: 0.4821

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.98      0.97     32424
           1       0.97      0.97      0.97     31029

    accuracy                           0.97     63453
   macro avg       0.97      0.97      0.97     63453
weighted avg       0.97      0.97      0.97     63453



## C. Interpretation

Using the final trained model:

• Print and examine the coefficients. Interpret at least two coefficients in context (you
can add a markdown cell to explain the relationships) 

• In a Markdown cell, state whether you think the model performs well or poorly, and
briefly explain why. 

### • Print and examine the coefficients. Interpret at least two coefficients in context (you can add a markdown cell to explain the relationships) 

In [20]:
#Taken from the code above
print("Model Coefficients:")
for feature, coef in zip(X.columns, final_model.coef_[0]): 
    print(f"{feature}: {coef:.4f}")

Model Coefficients:
month: -0.0195
arr_flights: 4.5190
carrier_ct: 26.8174
nas_delay: 47.1367
late_aircraft_ct: 37.0990
security_delay: 0.3130
weather_category: 0.4821


Looking at the final logistic regression model, each coefficient represents how that feature affects the likelihood of a flight having a high delay. For example, taking a look at the coefficient for nas_delay, we can see that it is 45.9578, which is a very large/positive coefficient. This means that as the NAS (National Airspace System) delay increases, the probability of a flight being classified as high delay increases sharply as well.

Another example would be taking a look at the coefficient for late_aircraft_ct which came out to be 36.1854. This number is also very large and positive. This indicates that if an aircraft has a history of flights delayed because the aircraft arrived late from a previous flight, then the the probability of a flight being classified as high delay increases too. We can see based off of the coefficients for these two features that they have a very large impact on our model.

### • Did the model performs well or poorly?

Based on the classification report, the model performs very well. The accuracy is 97%, and both precision and recall are above 95% for each class, resulting in an F1 score of 0.97. This indicates that the model is highly effective at correctly classifying both low-delay and high-delay flights, with fairly few misclassifications. Overall, the model appears reliable and strong for predicting the severity of flight delays using the selected features.

# Logistic Regression with L1 Regularization

Train a second logistic regression model using the L1 regularization using the same train-
test split

## A. Feature selection

Using the L1-regularized model and printing the coefficients, identify which features were
removed (coefficients shrunk to zero). Add a markdown cell and state which features were
removed.

In [21]:
# Logistic Regression with Lasso (L1) regularization

# Added more iterations and changed solver because I received a warning that liblinear couldn’t fully optimize the model in the default iterations
l1_model = LogisticRegression(penalty='l1', solver='saga', max_iter=5000) 
l1_model.fit(X_train_scaled, y_train_balanced)

# Print coefficients with feature names
print("L1-Regularized Model Coefficients:")
for feature, coef in zip(X.columns, l1_model.coef_[0]):
    print(f"{feature}: {coef:.4f}")

C:\Users\tahim\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\tahim\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


L1-Regularized Model Coefficients:
month: -0.0197
arr_flights: 4.4072
carrier_ct: 30.3610
nas_delay: 55.6252
late_aircraft_ct: 42.5229
security_delay: 0.3560
weather_category: 0.5367


After training the logistic regression model with L1 regularization, we can see that none of the coefficients were shrunk to zero, which means that no features were removed by the Lasso regularization.

This indicates that all selected features (month, arr_flights, carrier_ct, nas_delay, late_aircraft_ct, security_delay, and weather_category) provide meaningful information for predicting high flight delays. L1 regularization helped to confirm this as it didn't shrink any of our features to 0.

## B. Model evaluation

Printing out the model's performance then: 

Comparing the two models: 

• Did L1 regularization improve, worsen, or not change performance?

• How did removing features affect the model?


In [22]:
# Print coefficients with feature names for L1-Regularized model
print("L1-Regularized Model Coefficients:")
for feature, coef in zip(X.columns, l1_model.coef_[0]):
    print(f"{feature}: {coef:.4f}")

# Predict on the test set
y_pred_l1 = l1_model.predict(X_test_scaled)

# Classification report
print("\nClassification Report (L1-Regularized Model):")
print(classification_report(y_test, y_pred_l1))

L1-Regularized Model Coefficients:
month: -0.0197
arr_flights: 4.4072
carrier_ct: 30.3610
nas_delay: 55.6252
late_aircraft_ct: 42.5229
security_delay: 0.3560
weather_category: 0.5367

Classification Report (L1-Regularized Model):
              precision    recall  f1-score   support

           0       0.97      0.98      0.97     32424
           1       0.97      0.97      0.97     31029

    accuracy                           0.97     63453
   macro avg       0.97      0.97      0.97     63453
weighted avg       0.97      0.97      0.97     63453



### • Did L1 regularization improve, worsen, or not change performance?

Comparing the first logistic regression model with the L1-regularized model, we see that performance did not change. Both models achieved an accuracy of 97% and the same precision, recall, and F1 scores for both classes. This indicates that applying L1 regularization neither improved or worsened the predictive ability of the model on this dataset. The L1 model produced coefficients very close to the standard model even though they were adjusted slightly, confirming that the overall relationship between features and high flight delays remained consistent.

### • How did removing features affect the model?

When examining the coefficients of the L1-regularized model, none of the features were shrunk to zero, meaning that L1 regularization did not remove any features. This can mean that all selected features (month, arr_flights, carrier_ct, nas_delay, late_aircraft_ct, security_delay, and weather_category) contribute meaningfully to predicting high flight delays. Since no features were removed, feature selection had no impact here on the model’s performance. This confirms that the chosen set of predictors is good and each feature provides fairly useful information.

# Logistic Regression with L2 Regularization

• Training the model using L2 regularization 

• Printing the coefficients 

• Printing the model’s performance 

• Comparing the coefficients of this model to the coefficients of the L1 and baseline models 

• Comparing the performance of this model to the performances of the L1 and baseline models

## • Train the model using L2 regularization 

In [23]:
ridge_model = LogisticRegression(penalty='l2', solver='liblinear')
ridge_model.fit(X_train_scaled, y_train_balanced)

C:\Users\tahim\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'l2'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass`

## • Printing the coefficients 

In [24]:
print("L2-Regularized Model Coefficients:")
for feature, coef in zip(X.columns, ridge_model.coef_[0]):
    print(f"{feature}: {coef:.4f}")

L2-Regularized Model Coefficients:
month: -0.0196
arr_flights: 4.3033
carrier_ct: 26.0893
nas_delay: 45.9551
late_aircraft_ct: 35.8692
security_delay: 0.3136
weather_category: 0.4717


## • Printing the model’s performance 

In [25]:
# Predict on the test set
y_pred_ridge = ridge_model.predict(X_test_scaled)

# Classification report
print("\nClassification Report (L2-Regularized Model):")
print(classification_report(y_test, y_pred_ridge))


Classification Report (L2-Regularized Model):
              precision    recall  f1-score   support

           0       0.97      0.98      0.97     32424
           1       0.97      0.97      0.97     31029

    accuracy                           0.97     63453
   macro avg       0.97      0.97      0.97     63453
weighted avg       0.97      0.97      0.97     63453



## • Comparing the coefficients of this model to the coefficients of the L1 and baseline models 

The coefficients of the L2-regularized (Ridge) model are very similar to those of the baseline and L1 models. Features like nas_delay, late_aircraft_ct, carrier_ct, and arr_flights still have the largest positive coefficients, showing that they are the strongest predictors of high flight delays.

Compared to the baseline model, most coefficients slightly decreased in magnitude, such as carrier_ct, nas_delay, late_aircraft_ct, and security_delay, while a few, like arr_flights and weather_category, slightly increased. Compared to the L1 model, the pattern is similar. L2 slightly shrinks most coefficients but does not eliminate any features since it is L2. Overall, Ridge regularization slightly adjusts feature strengths without changing the ranking or removing features. This helped to maintain the overall importance pattern seen in the baseline model.

## • Comparing the performance of this model to the performances of the L1 and baseline models

The L2-regularized logistic regression model showed performance metrics seemingly the same as both the baseline and L1-regularized models. All three models reached an accuracy of 97% on the test set, with similar precision, recall, and F1 scores for both the low-delay and high-delay classes.

This indicates that applying L2 regularization did not improve or worsen overall predictive performance on this dataset. The slight reduction in the size of coefficients from L2 regularization does not significantly impact the model’s ability to classify high and low flight delays.

All three models performed equally well, confirming that the chosen features are strong predictors and that regularization mainly affects coefficient shrinkage rather than predictive performance in this case.

# Logistic Regression with Elastic Net Regularization

• Training the model using Elastic Net regularization 

• Printing the coefficients 

• Printing the model’s performance 

• Comparing the coefficients of this model to the coefficients of the L1, L2 and baseline models 

• Comparing the performance of this model to the performances of the L1, L2 and baseline models

## • Train the model using Elastic Net regularization 

In [26]:
elastic_net_model = LogisticRegression( penalty='elasticnet', solver='saga', l1_ratio=0.5)

elastic_net_model.fit(X_train_scaled, y_train_balanced)

C:\Users\tahim\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\tahim\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'elasticnet'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.5
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

## • Print the coefficients 

In [27]:
print("Elastic Net Model Coefficients:")
for feature, coef in zip(X.columns, elastic_net_model.coef_[0]):
    print(f"{feature}: {coef:.4f}")

Elastic Net Model Coefficients:
month: -0.0259
arr_flights: 5.9284
carrier_ct: 16.4534
nas_delay: 17.6071
late_aircraft_ct: 18.6615
security_delay: 0.2176
weather_category: 0.3494


## • Printing the model’s performance 

In [28]:
# Predict on the test set
y_pred_en = elastic_net_model.predict(X_test_scaled)

# Classification report
print("\nClassification Report (Elastic Net Model):")
print(classification_report(y_test, y_pred_en))


Classification Report (Elastic Net Model):
              precision    recall  f1-score   support

           0       0.96      0.97      0.97     32424
           1       0.97      0.95      0.96     31029

    accuracy                           0.96     63453
   macro avg       0.97      0.96      0.96     63453
weighted avg       0.96      0.96      0.96     63453



## • Comparing the coefficients of this model to the coefficients of the L1, L2 and baseline models 

The coefficients of the Elastic Net logistic regression model show noticeable differences compared to the baseline, L1, and L2 models. Features like nas_delay, late_aircraft_ct, and carrier_ct still have the largest positive coefficients, but their sizes are much smaller than in the previous models. This shows the combined shrinking effect of L1 and L2 regularization.

Compared to the baseline model, most coefficients decreased in magnitude especially carrier_ct, nas_delay, and late_aircraft_ct while arr_flights slightly increased. Compared to the L1 model, Elastic Net shrinks all coefficients more uniformly without shrinking any to zero. Compared to the L2 model, the coefficients were somewhat smaller showing that the L1 part tries to shrink some coefficients more, while the L2 part keeps the values more stable.. Overall, Elastic Net maintains the relative importance of features but reduces the influence of the strongest predictors and creating some sort of balance within these models.

## • Comparing the performance of this model to the performances of the L1, L2 and baseline models